# Étape 6 - Incertitudes de modélisation (MultiModel)

Une prévision sans marge d'erreur est peu exploitable. Méthode simple pour obtenir
une incertitude : **perturber à la fois les données et le modèle**, et regarder
la dispersion des prédictions.

## La classe `MultiModel`

Elle entraîne `n_models` copies du modèle. Chaque copie diffère par :

- un **échantillon bootstrap** différent des données (tirage avec remise)
- une **graine aléatoire** différente

C'est un *décorateur* : il enveloppe n'importe quel estimateur scikit-learn.
Il hérite aussi de `PythonModel` pour être compatible **MLflow** (étape suivante du projet).

Conséquences pratiques :

- `predict` prend un argument supplémentaire `context` (imposé par MLflow) → `model.predict(None, X)`
- `predict` renvoie un dataframe avec une colonne par copie (`y_pred_0` … `y_pred_9`) plus `y_pred_simple`
- `plotly_predictions` détecte ce cas et trace une **plage** au lieu d'une courbe

## 1. Importer les librairies

In [ ]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

## 2. Reprendre les étapes 1 à 4

In [ ]:
# --- Reprise des étapes 1 à 4 ---
# jeu d'entraînement
df = etl(settings.DATA_DIR, 197, 200)
df = features_offline(df)
x_train = df.drop(columns=['cash_in']).set_index('order_date')
y_train = df[['order_date', 'cash_in']].set_index('order_date')['cash_in']

# jeu de prédiction
past = etl(settings.DATA_DIR, 200, 200)
future = span_future(past['order_date'].max())
future = features_online(future, past)
future = future.set_index('order_date')

# modèle simple entraîné sur tout
simple_model = RandomForestRegressor(n_estimators=10, random_state=42)
simple_model.fit(x_train, y_train)
future.head()

## 3. Regarder la classe `MultiModel`

In [ ]:
MultiModel?

## 4. Créer un MultiModel de 10 répliques

On enveloppe `simple_model` (le `RandomForestRegressor` créé au-dessus).

In [ ]:
multi_model = MultiModel(simple_model, n_models=10)
multi_model

## 5. Validation croisée temporelle (3 folds)

Même appel que pour le modèle simple : `cross_validate` gère les deux cas.

In [ ]:
maes, preds = cross_validate(multi_model, x_train, y_train, n_fold=3)
maes

## 6. Moyenne et écart-type des MAEs par fold

`maes` a une ligne par fold et une colonne par réplique. On agrège **par ligne** (`axis=1`).

In [ ]:
pd.DataFrame({
    'mae_moyenne': maes.mean(axis=1),
    'mae_ecart_type': maes.std(axis=1),
})

## 7. Tracer les prédictions de validation croisée (plage)

In [ ]:
plotly_predictions(preds, y_train)

## 8. Regarder le code de `MultiModel.fit`

In [ ]:
MultiModel.fit??

## 9. Entraîner le MultiModel sur tout le jeu d'entraînement

In [ ]:
multi_model.fit(x_train, y_train)

## 10. Regarder le code de `MultiModel.predict`

In [ ]:
MultiModel.predict??

## 11. Prédire le chiffre d'affaires futur

Attention à l'API : `predict(context, X)`. On passe `None` comme `context`.

In [ ]:
y_pred = multi_model.predict(None, future)
y_pred.head(20)

## 12. Tracer la prévision avec incertitude

`plotly_predictions` trace la plage min/max des 10 répliques : la largeur de la bande
représente l'incertitude du modèle.

In [ ]:
plotly_predictions(y_pred)

## Félicitations 🎉

Tu as parcouru tout le pipeline : **ETL → features offline → entraînement + validation
temporelle → features online → prévision → incertitude par bootstrap**.

Le `MultiModel` obtenu est déjà au format MLflow : le notebook suivant du projet
(`notebooks/mlflow_tracking.ipynb`) ajoute le *tracking*, la reproductibilité et le packaging.